PART 05：为什么只给一个 bash？——"One loop & Bash is all you need"
代码跑通了，留一个最值得琢磨的问题收尾。

你的直觉可能是反对的
看到工具定义那段，你八成想过：就给一个 bash？为什么不给它 20 个精细工具——read_file、write_file、search_code、run_python、list_dir……每个工具参数结构化、职责单一，听起来工程上漂亮多了。

我之前自己写过一个小 Agent，就是这么干的：挂了 7 个精细工具，查订单、改地址、查物流，每个都单独定义。运行得也挺好。

所以第一次看清 Claude Code 的内核只给一个 bash 时，我是愣了一下的。想明白之后，服了。

bash 不是一把刀，是一台机床
想给它 20 个工具，先算一笔账：

读文件，bash 里是 cat
写文件，是 echo ... > file
搜代码，是 grep -rn
找文件，是 find
跑脚本，是 python xxx.py
组合操作，是管道 |、重定向 >、链式 &&
一个 bash，吞掉了你打算手写的整个工具箱。

而且 shell 的能力空间是组合爆炸的——你永远枚举不完用户会遇到的操作，但任意操作几乎都能用几条 shell 命令现凑出来。专用工具是 20 把固定的刀，bash 是一台能现加工任何刀具的机床。

换来两样更值钱的东西：

一，模型的选择负担最小。 工具列表越短，模型"该用哪个"的决策越干净，选错工具的概率越低。给它 20 个长得差不多的工具，它会在 read_file 和 search_code 之间犹豫，然后选错。


二，能力上限是整个 shell 生态。 精细工具的能力上限是你写工具那天的想象力，bash 的上限是几十年积累的全部命令行工具。


当然这不是说专用工具没价值——参数结构化、好审计、行为可控，这些是 bash 给不了的。Claude Code 的选择是：底座用 bash 换最大灵活性，再靠一层层机制把"可控"补回来。 你后面会看到权限系统怎么补安全、专门的读写工具怎么补精度。


先有自由，再上枷锁，顺序不能反。


它现在还很糙——这份"糙"就是后面的目录
最后诚实地列一下这个 142 行内核的缺陷，每个缺陷都对应往后的一篇：

黑名单挡不住 rm -rf ~/重要目录
 → 需要权限门禁：工具执行前先过审批
一条 find / 的输出能撑爆上下文
，聊久了账本无限变厚 → 需要上下文压缩
长任务会走神
，干着干着忘了最初的目标 → 需要 Todo 计划机制
所有中间过程全记在一本账上
，查个资料把主上下文弄脏 → 需要子代理隔离
一次只能干一件事
，慢命令一跑全卡住 → 需要后台任务
会话关了就全忘
 → 需要跨会话记忆
这个模块的路线就此清晰了：循环从第一天写完就再也不变，变的是外面一圈圈长出来的壳——权限、工具分发、Hooks、计划、子代理、上下文压缩、记忆、任务系统、后台任务、定时调度、团队协作、插件、集成、编排、目标闭环。每一篇拆一个器官，拆完你就拥有一个自己完全理解的 Claude Code。

给你的学习建议
别只看。把这 142 行亲手敲一遍（不要复制粘贴），跑通上面三个实录。敲的过程中你大概率会在 tool_use_id 配对上报错一次——那一次报错，比看十篇文章都管用。